<a href="https://colab.research.google.com/github/eunmindow-star/chaeeunmin/blob/main/code/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==============================================================================
# [1. 노트북 준비] 10주차 노트북 불러오기, 변수 만들기, in_analysis 코드
# ==============================================================================
drive.mount('/content/drive')
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')
print(f"전체 원시자료 행 수: {len(df):,}명")

# 10주차 기본 파생변수 생성 (전체 54,170명 표본설계 구조 유지)
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notnull(), 0, np.nan))
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notnull(), 0, np.nan))

# 10주차 Table 1 기본 분석 대상자 (in_analysis)
df['in_analysis'] = (df['domain_hs'] == 1) & df['bf_skip'].notnull() & df['obesity'].notnull()
n_table1 = int(df['in_analysis'].sum())
print(f"10주차 Table 1 기본 분석 대상(in_analysis) 인원: {n_table1:,}명")

# ==============================================================================
# [2. 11주차 공변량 및 매개변수 전처리]
# ==============================================================================
# 교란 후보 (Confounders): 성별(SEX), 학년(GRADE), 주관적 경제상태(E_SES)
df['female'] = np.where(df['SEX'] == 2, 1, np.where(df['SEX'] == 1, 0, np.nan))
df['inc_grp'] = np.select(
    [df['E_SES'].isin([1, 2]), df['E_SES'] == 3, df['E_SES'].isin([4, 5])],
    [1, 2, 3], default=np.nan
)

# 매개 후보 (Mediators - Model 3 비교용): 수면충족감(M_SLP_EN), 신체활동(PA_TOT), 야식(F_NE)
df['slp_grp'] = np.select(
    [df['M_SLP_EN'].isin([1, 2]), df['M_SLP_EN'] == 3, df['M_SLP_EN'].isin([4, 5])],
    [1, 2, 3], default=np.nan
)
df['pa_act'] = np.where(df['PA_TOT'].isin([4, 5, 6, 7, 8]), 1,
                        np.where(df['PA_TOT'].isin([1, 2, 3]), 0, np.nan))
df['night_eat'] = np.where(df['F_NE'].isin([4, 5, 6, 7, 8]), 1,
                           np.where(df['F_NE'].isin([1, 2, 3]), 0, np.nan))

# ==============================================================================
# [3. 완전 사례(Complete Case) 표시 및 Table 1 인원 차이 확인]
# ==============================================================================
confounders = ['female', 'GRADE', 'inc_grp']
df['complete_case'] = df['in_analysis'] & df[confounders].notna().all(axis=1)

mediators = ['slp_grp', 'pa_act', 'night_eat']
df['complete_case_m3'] = df['complete_case'] & df[mediators].notna().all(axis=1)

n_cc = int(df['complete_case'].sum())
diff_n = n_table1 - n_cc

print("\n=== [완전 사례(Complete Case) 검증 결과] ===")
print(f"1) Table 1 분석 대상(in_analysis): {n_table1:,}명")
print(f"2) 보정변수 완전 사례(complete_case): {n_cc:,}명")
print(f"3) 인원 차이 (Table 1 대상 - 완전 사례): {diff_n:,}명")

# ==============================================================================
# [4. C-2 회귀: 복합표본 도메인 로지스틱 회귀분석 (자료를 자르지 않고 도메인 처리)]
# ==============================================================================
def fit_domain_logit(data, formula, domain_mask, weight_col='W', strata_col='STRATA', cluster_col='CLUSTER'):
    model_vars = [v.strip() for v in formula.replace('~', '+').replace('C(', '').replace(')', '').split('+')]
    n_used = int(domain_mask.sum())

    # 분석 대상 밖의 사람은 행을 지우지 않고 가중치를 1e-12로 설정하여 설계(STRATA, CLUSTER) 구조 유지
    w_domain = np.where(domain_mask, data[weight_col], 1e-12)

    temp_df = data.copy()
    for v in model_vars:
        temp_df[v] = temp_df[v].fillna(temp_df[v].mode()[0])

    psu_group = temp_df[strata_col].astype(str) + "_" + temp_df[cluster_col].astype(str)

    model = smf.glm(
        formula=formula,
        data=temp_df,
        family=sm.families.Binomial(),
        var_weights=w_domain
    ).fit(cov_type='cluster', cov_kwds={'groups': psu_group})

    coef = model.params['bf_skip']
    se = model.bse['bf_skip']
    pval = model.pvalues['bf_skip']
    or_val = np.exp(coef)
    ci_low = np.exp(coef - 1.96 * se)
    ci_high = np.exp(coef + 1.96 * se)

    return {
        'n': n_used,
        'coef': coef,
        'OR': or_val,
        'CI_low': ci_low,
        'CI_high': ci_high,
        'p_value': pval,
        'OR_CI': f"{or_val:.2f} ({ci_low:.2f}–{ci_high:.2f})"
    }

# 모형 1(노출만)과 모형 2(보정집합)를 반드시 '같은 사람(df['complete_case'])'으로 적합
m1_res = fit_domain_logit(df, "obesity ~ bf_skip", df['complete_case'])
m2_res = fit_domain_logit(df, "obesity ~ bf_skip + female + C(GRADE) + C(inc_grp)", df['complete_case'])
m3_res = fit_domain_logit(df, "obesity ~ bf_skip + female + C(GRADE) + C(inc_grp) + C(slp_grp) + pa_act + night_eat", df['complete_case_m3'])

table2_df = pd.DataFrame([
    {
        '모형 (Model)': 'Model 1 (노출만)',
        '보정 변수': '없음 (Unadjusted)',
        '분석 인원 (n)': f"{m1_res['n']:,}",
        '회귀계수 (β)': f"{m1_res['coef']:.4f}",
        'OR (95% CI)': m1_res['OR_CI'],
        'p-value': f"{m1_res['p_value']:.4f}"
    },
    {
        '모형 (Model)': 'Model 2 (주 모형: 보정집합)',
        '보정 변수': '성별(SEX), 학년(GRADE), 경제상태(E_SES)',
        '분석 인원 (n)': f"{m2_res['n']:,}",
        '회귀계수 (β)': f"{m2_res['coef']:.4f}",
        'OR (95% CI)': m2_res['OR_CI'],
        'p-value': f"{m2_res['p_value']:.4f}"
    },
    {
        '모형 (Model)': 'Model 3 (과보정 비교: 매개 포함)',
        '보정 변수': 'Model 2 + 수면(M_SLP_EN), 신체활동(PA_TOT), 야식(F_NE)',
        '분석 인원 (n)': f"{m3_res['n']:,}",
        '회귀계수 (β)': f"{m3_res['coef']:.4f}",
        'OR (95% CI)': m3_res['OR_CI'],
        'p-value': f"{m3_res['p_value']:.4f}"
    }
])

print("\n=== [C-2 Table 2: 복합표본 도메인 로지스틱 회귀분석 결과] ===")
display(table2_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
전체 원시자료 행 수: 54,170명
10주차 Table 1 기본 분석 대상(in_analysis) 인원: 25,077명

=== [완전 사례(Complete Case) 검증 결과] ===
1) Table 1 분석 대상(in_analysis): 25,077명
2) 보정변수 완전 사례(complete_case): 25,070명
3) 인원 차이 (Table 1 대상 - 완전 사례): 7명

=== [C-2 Table 2: 복합표본 도메인 로지스틱 회귀분석 결과] ===


,모형 (Model),보정 변수,분석 인원 (n),회귀계수 (β),OR (95% CI),p-value
0,Model 1 (노출만),없음 (Unadjusted),"25,070",0.0687,1.07 (1.00–1.15),0.0562
1,Model 2 (주 모형: 보정집합),"성별(SEX), 학년(GRADE), 경제상태(E_SES)","25,070",0.0863,1.09 (1.01–1.17),0.0215
2,Model 3 (과보정 비교: 매개 포함),"Model 2 + 수면(M_SLP_EN), 신체활동(PA_TOT), 야식(F_NE)","25,070",0.0941,1.10 (1.02–1.18),0.0127
